# Notebook 2: Retrieval of passages for the questions

This notebook prepares the context used in Notebook 3. It loads the fixed dev (150) and test (300) ARC-Challenge questions, builds a BM25 index over the ARC Corpus, compares three ways of writing the search query, and saves the top 3 retrieved passages for every question.

## 1. Setup
Installs the libraries, mounts Google Drive and loads the fixed question ids from `question_sets.json`.

In [1]:
!pip -q install datasets bm25s PyStemmer
from google.colab import drive
drive.mount('/content/drive')
import os, json, gc, time, shutil, zipfile, glob, itertools
import bm25s, Stemmer

BASE = "/content/drive/MyDrive/hallu_project"
qs = json.load(open(f"{BASE}/question_sets.json"))

Mounted at /content/drive


## 2. Resolve questions and save them
Loads ARC-Challenge once, looks up the question text, options and correct answer for every dev and test id, and saves them to `questions_resolved.json`.

In [2]:
from datasets import load_dataset
arc_ds = load_dataset("allenai/ai2_arc", "ARC-Challenge")
arc = {r["id"]: r for sp in ["validation", "test"] for r in arc_ds[sp]}

resolved = {n: [{"id": i, "question": arc[i]["question"],
                 "choices": arc[i]["choices"], "gold": arc[i]["answerKey"]}
               for i in qs[n]] for n in ["dev", "test"]}
json.dump(resolved, open(f"{BASE}/questions_resolved.json", "w"), indent=1)
print({k: len(v) for k, v in resolved.items()})   # {'dev': 150, 'test': 300}

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

ARC-Challenge/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

ARC-Challenge/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 55.7kB            

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

{'dev': 150, 'test': 300}


## 3. Prepare the ARC Corpus
Copies the ARC corpus zip from Drive to local storage (faster and more reliable), extracts it and finds the path of `ARC_Corpus.txt`.

In [3]:
zpath = f"{BASE}/data/ARC-V1-Feb2018.zip"
for attempt in range(3):
    try:
        shutil.copyfile(zpath, "/content/arc.zip"); break
    except OSError as e:
        print("retry", attempt + 1, e); time.sleep(5)

os.makedirs("/content/arc", exist_ok=True)
zipfile.ZipFile("/content/arc.zip").extractall("/content/arc")
CORPUS = glob.glob("/content/arc/**/ARC_Corpus.txt", recursive=True)[0]
print(CORPUS, os.path.getsize(CORPUS) // 1024 // 1024, "MB")

/content/arc/ARC-V1-Feb2018-2/ARC_Corpus.txt 1415 MB


## 4. Retrieval helpers
`build_index(n)` builds a BM25 index over the first n lines of the corpus. `retr` returns the top-k passages for a list of queries. `rates` measures how often the text of the correct option (gold rate) and of the wrong options appears in the retrieved passages.

In [4]:
stemmer = Stemmer.Stemmer("english")
docs, retriever, CURRENT_N = None, None, None

def build_index(n_docs):
    global docs, retriever, CURRENT_N
    docs = None; retriever = None; gc.collect()
    with open(CORPUS, encoding="utf-8", errors="ignore") as f:
        docs = [l.strip() for l in itertools.islice(f, n_docs) if l.strip()]
    retriever = bm25s.BM25()
    retriever.index(bm25s.tokenize(docs, stopwords="en", stemmer=stemmer))
    CURRENT_N = n_docs
    print("docs:", len(docs))

def retr(queries, k):
    res, _ = retriever.retrieve(bm25s.tokenize(queries, stopwords="en", stemmer=stemmer), corpus=docs, k=k)
    return res

def rates(ctx_by_id, qlist):
    g = w = nw = 0
    for q in qlist:
        p = ctx_by_id[q["id"]].lower()
        for t, l in zip(q["choices"]["text"], q["choices"]["label"]):
            hit = t.lower() in p
            if l == q["gold"]: g += hit
            else: w += hit; nw += 1
    return round(g / len(qlist), 3), round(w / nw, 3)   # gold rate, avg wrong-option rate

## 5. Compare three query styles (2M lines, dev set)
Builds an index over the first 2 million corpus lines and tests three queries on the dev set: A = question + options, B = question only, C = one query per option (top 1 each). The best style is the one with the highest gold rate.

In [5]:
build_index(2_000_000)
qlist = resolved["dev"]
ctx = {}

# A: question + options
res = retr([q["question"] + " " + " ".join(q["choices"]["text"]) for q in qlist], 3)
ctx["A"] = {q["id"]: "\n".join(res[i]) for i, q in enumerate(qlist)}

# B: question only
res = retr([q["question"] for q in qlist], 3)
ctx["B"] = {q["id"]: "\n".join(res[i]) for i, q in enumerate(qlist)}

# C: one query per option, top 1 each
flat = [(i, t) for i, q in enumerate(qlist) for t in q["choices"]["text"]]
res = retr([qlist[i]["question"] + " " + t for i, t in flat], 1)
tmp = {}
for (i, _), r in zip(flat, res):
    tmp.setdefault(qlist[i]["id"], []).append(r[0])
ctx["C"] = {k: "\n".join(v) for k, v in tmp.items()}

for name in "ABC":
    print(name, "gold rate, avg wrong-option rate:", rates(ctx[name], qlist))

Split strings:   0%|          | 0/2000000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/2000000 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/2000000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2000000 [00:00<?, ?it/s]

docs: 2000000


... (6 outputs left out)

Split strings:   0%|          | 0/598 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/598 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/598 [00:00<?, ?it/s]

A gold rate, avg wrong-option rate: (0.113, 0.136)
B gold rate, avg wrong-option rate: (0.013, 0.013)
C gold rate, avg wrong-option rate: (0.053, 0.065)


## 6. Larger index (5M lines, query style A)
Rebuilds the index over the first 5 million corpus lines and repeats query style A on the dev set, to see whether a larger corpus finds the correct option text more often.

In [6]:
build_index(5_000_000)
qlist = resolved["dev"]
res = retr([q["question"] + " " + " ".join(q["choices"]["text"]) for q in qlist], 3)
ctxA = {q["id"]: "\n".join(res[i]) for i, q in enumerate(qlist)}
print("A @5M gold rate, avg wrong-option rate:", rates(ctxA, qlist))

Split strings:   0%|          | 0/5000000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/5000000 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/5000000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/5000000 [00:00<?, ?it/s]

docs: 5000000


Split strings:   0%|          | 0/150 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/150 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/150 [00:02<?, ?it/s]

A @5M gold rate, avg wrong-option rate: (0.113, 0.109)


## 7. Final retrieval and saving
Uses query style A with the 5M-line index (top 3 passages per question) to build the contexts for the dev and test sets, and saves them as `retrieved_dev.json` and `retrieved_test.json`. It also saves, for every question, whether the correct option text appears in the passages (`retrieval_hits_*.json`) and prints the overall gold-hit rate.

In [7]:
N_FINAL = 5_000_000
if CURRENT_N != N_FINAL:
    build_index(N_FINAL)

def build_ctx(qlist):
    res = retr([q["question"] + " " + " ".join(q["choices"]["text"]) for q in qlist], 3)
    return {q["id"]: "\n".join(f"- {p}" for p in res[i]) for i, q in enumerate(qlist)}

for split in ["dev", "test"]:
    ql = resolved[split]
    out = build_ctx(ql)
    json.dump(out, open(f"{BASE}/retrieved_{split}.json", "w"), indent=1)
    gold_hits = {}
    for q in ql:
        gold_text = q["choices"]["text"][q["choices"]["label"].index(q["gold"])].lower()
        gold_hits[q["id"]] = gold_text in out[q["id"]].lower()
    json.dump(gold_hits, open(f"{BASE}/retrieval_hits_{split}.json", "w"))
    print(split, "gold-hit rate:", round(sum(gold_hits.values()) / len(gold_hits), 3))

Split strings:   0%|          | 0/150 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/150 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/150 [00:00<?, ?it/s]

dev gold-hit rate: 0.113


Split strings:   0%|          | 0/300 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/300 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/300 [00:00<?, ?it/s]

test gold-hit rate: 0.107
